# Hindustani Raga Classifier — per-raga HMMs (Tansen-style)

**Why this exists**: real academic precedent for exactly this — Pandey, Mishra & Ipe (2003), *"Tansen: A System for Automatic Raga Identification"* (cited in the TRF paper's own literature review), models raga identification as one HMM per raga, trained generatively on that raga's note sequences, classified by Bayes' rule (which raga's HMM assigns highest posterior to this sequence). A different modeling paradigm from `raga_classifier_pitch_contour.ipynb`'s single discriminative GRU — worth comparing empirically, not assuming either is better.

**Why it might specifically suit our situation**: a set of permitted melodic transitions and characteristic phrases (arohana/avarohana, pakad) is close to literally what an HMM's states and transition matrix model. It also has far fewer parameters than even the shrunk GRU (61 independent small models vs. one 364K-param network) — continuing this project's running theme that data scarcity, not model capacity, has been the real constraint.

**Real caveat, not just an upside**: 61 *independent* per-class HMMs share zero parameters across classes, unlike the GRU's shared embedding/RNN — so while each HMM is individually tiny, the system as a whole isn't strictly more sample-efficient in aggregate. This is exactly the kind of thing to let the held-out numbers decide, not assume.

**No GPU needed** — `hmmlearn`'s Baum-Welch (EM) training is CPU-only and fast for this scale. Notebook Settings can skip the accelerator entirely, saving GPU quota.

**Reuses the exact same cached token sequences** as the pitch-contour GRU notebook — same quantization scheme, same config keys in the cache hash. If that notebook's run has completed, point `EXISTING_CACHE_INPUT` below at its Save Version output (Add Data → Notebook Output Files) to skip precompute entirely, same mechanism as before. If not, this notebook computes its own from scratch (self-sufficient either way).

**Saves per-segment log-likelihoods for all 61 classes on the test set** (not just the final accuracy) — needed later for the hybrid/ensemble idea (combine this model's and the GRU's predictions), once both have real standalone results to combine.

In [ ]:
# --- Config — cache-relevant values MUST match raga_classifier_pitch_contour.ipynb
# exactly for EXISTING_CACHE_INPUT reuse to work (same hash key formula).
# TRAIN_SEGMENT_STRIDE/EVAL_SEGMENT_STRIDE (2026-09-18, renamed+split from a
# single SEGMENT_STRIDE) match that notebook's train/eval decoupling — see
# below. MAX_SEGMENTS_PER_CLASS below already bounds per-model HMM training
# cost independently, so this doesn't blow up training time here even
# though more segments/file now exist for training. ---
import os
# Must run before numpy is imported ANYWHERE in this kernel (first happens in
# the precompute cell) — otherwise OpenBLAS/MKL/OMP each spin up their own
# internal thread pool per process, and HMM_TRAIN_WORKERS worker processes
# each doing that oversubscribes the CPU, silently eating most of the
# multiprocessing speedup added 2026-09-18 below. Each per-class HMM fit is
# small dense math (12 states) — the real parallelism we want is ACROSS the
# 61 independent per-class models, not within one model's matrix ops.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

AUDIO_SAMPLE_RATE = 44100
MELODY_HOP_SIZE = 128
QUANT_K = 5
VOCAB_SIZE = 209
INPUT_LENGTH = 5000

# TRAIN vs EVAL windowing DECOUPLED (2026-09-18) — ported from the GRU
# notebook after real evidence there: applying overlapping windows uniformly
# to train AND val/test segments dropped majority-vote test accuracy
# (41.0% -> 32.8%) even though per-segment accuracy improved slightly.
# Root cause: 50%-overlapping windows share up to half their tokens with
# their neighbors, so their prediction errors are correlated, not
# independent — majority voting's benefit comes specifically from averaging
# INDEPENDENT errors, so correlated "votes" undermine it. This notebook's
# own test-set majority vote (cell-8) was exposed to the exact same issue
# (same overlapping cache, same aggregation-by-vote), so the same fix
# applies here even though this run hadn't gotten as far as testing yet.
# Overlap stays a real win for TRAINING (more augmented examples, no
# independence requirement there).
TRAIN_SEGMENT_STRIDE = INPUT_LENGTH // 2   # 50% overlap — augmentation only
EVAL_SEGMENT_STRIDE = INPUT_LENGTH         # NO overlap — independent votes for majority voting

MIN_VOICED_FRACTION = 0.5
MAX_SEGMENTS_PER_FILE = 50
MAX_ANALYZE_SECONDS = 480
INTRO_SKIP_SECONDS = 20
OUTRO_SKIP_SECONDS = 15
PRECOMPUTE_WORKERS = 4

# HMM-specific config
N_HMM_STATES = 12    # rough analogue of 12 chromatic semitone positions per
                      # octave — a musically-motivated starting point, not
                      # tuned; states represent latent "functional" pitch
                      # categories, not the 209 fine-grained cents tokens directly
HMM_ITER = 100        # Baum-Welch (EM) iterations

# Parallelize the 61 independent per-class HMM fits across processes (added
# 2026-09-18, see the training cell for the full story): a live run showed
# ~1,178s/model even for the smallest classes trained first, extrapolating
# to 20+ hours serial for all 61 — far more than any Kaggle session survives.
# GPU/TPU is NOT the right lever here: hmmlearn has no GPU support at all,
# and Baum-Welch's sequential per-timestep recursion isn't a natural fit for
# GPU/TPU without a from-scratch batched-tensor reimplementation. Since each
# raga's HMM shares no parameters with any other, running fits concurrently
# across CPU processes is the correct, zero-algorithmic-risk lever instead —
# reuses PRECOMPUTE_WORKERS' core count, already proven on this Kaggle CPU tier.
HMM_TRAIN_WORKERS = PRECOMPUTE_WORKERS

SEED = 42             # SAME seed as both other notebooks — keeps the file-level
                      # split identical across all three for a fair comparison

SEGMENTS_CACHE_DIR = "/kaggle/working/pitch_cache"
MODELS_PATH = "/kaggle/working/hmm_models.pkl"

# Point at the pitch-contour GRU notebook's Save Version output (Add Data >
# Notebook Output Files) once that run has completed, to skip precompute
# entirely — same cache, same config, directly reusable. Leave None to
# compute fresh (self-sufficient, just slower).
EXISTING_CACHE_INPUT = None

# Point at a PARTIAL hmm_models.pkl from an earlier (interrupted) run of
# THIS notebook (Add Data > Notebook Output Files) to skip already-trained
# classes and resume the rest. Added 2026-09-18 alongside incremental
# checkpointing in the training cell — before this, a session timeout meant
# total loss even with several models already trained, since the old code
# only saved once, after the entire loop finished.
EXISTING_MODELS_INPUT = None

In [ ]:
import subprocess, time, os

def run(cmd, timeout, label=None):
    label = label or cmd
    print(f"--- RUNNING ({timeout}s timeout): {label}")
    t0 = time.time()
    try:
        result = subprocess.run(cmd, shell=True, timeout=timeout,
                                 stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except subprocess.TimeoutExpired as e:
        print(e.stdout or "")
        raise RuntimeError(f"TIMED OUT after {time.time()-t0:.0f}s: {label}")
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"FAILED (exit {result.returncode}): {label}")
    print(f"--- OK ({time.time()-t0:.0f}s): {label}")
    return result

run("apt-get -qq install -y ffmpeg", timeout=120, label="apt-get ffmpeg")
# essentia only actually needed if precompute runs from scratch (no cache
# attached) — installed unconditionally anyway since it's cheap and this
# notebook should work standalone. Same numpy-ABI defense applied proactively
# this time (learned the hard way in the GRU notebook, not repeating that mistake).
run("pip install -q essentia hmmlearn", timeout=300, label="pip install essentia + hmmlearn")
_pip_show = run("pip show numpy", timeout=30, label="read resolved numpy version")
_numpy_version = next(
    line.split(":", 1)[1].strip()
    for line in _pip_show.stdout.splitlines() if line.startswith("Version:")
)
print(f"Resolved numpy version: {_numpy_version}")
run(f"pip install -q --force-reinstall --no-deps numpy=={_numpy_version}", timeout=120,
    label="force-reinstall numpy binaries (pinned)")
run(
    'python -c "import numpy, sklearn, hmmlearn, essentia.standard as es; '
    'from hmmlearn import hmm; hmm.CategoricalHMM(n_components=4); '
    'es.TonicIndianArtMusic(); es.PredominantPitchMelodia(); '
    'print(numpy.__version__, hmmlearn.__version__, \'OK\')"',
    timeout=60,
    label="verify numpy/sklearn/hmmlearn/essentia all import and load cleanly",
)
os.makedirs(SEGMENTS_CACHE_DIR, exist_ok=True)

In [ ]:
import shutil

if EXISTING_CACHE_INPUT:
    if not os.path.isdir(EXISTING_CACHE_INPUT):
        raise RuntimeError(f"EXISTING_CACHE_INPUT={EXISTING_CACHE_INPUT!r} not found.")
    shutil.copytree(EXISTING_CACHE_INPUT, SEGMENTS_CACHE_DIR, dirs_exist_ok=True)
    print(f"Copied in {len(os.listdir(SEGMENTS_CACHE_DIR))} cached files from a prior session.")
else:
    print("No existing cache attached — starting precompute from scratch.")

In [ ]:
# --- Locate the dataset and enumerate every recording — identical to the
# other two notebooks, so all three are comparable. ---
import glob
from collections import defaultdict

candidates = [c for c in glob.glob("/kaggle/input/**/Thaat and Raga Forest*", recursive=True) if os.path.isdir(c)]
if not candidates:
    raise RuntimeError("TRF dataset not found under /kaggle/input. Add Data > 'suryamajumder/thaat-and-raga-forest-trf-dataset'.")
DATASET_ROOT = candidates[0]
print("Dataset root:", DATASET_ROOT)

files_by_raga = defaultdict(list)
for path in glob.glob(os.path.join(DATASET_ROOT, "*", "*", "*.mp3")):
    parts = path.split(os.sep)
    files_by_raga[(parts[-3], parts[-2])].append(path)

ragas = sorted(files_by_raga.keys(), key=lambda k: k[1])
raga2idx = {raga_key: i for i, raga_key in enumerate(ragas)}
idx2raga = {i: {"thaat": t, "raga": r} for (t, r), i in raga2idx.items()}
print(f"{len(ragas)} ragas, {sum(len(v) for v in files_by_raga.values())} recordings total")

In [ ]:
import random
random.seed(SEED)

train_files, val_files, test_files = [], [], []
for raga_key, fs in files_by_raga.items():
    fs = sorted(fs)
    random.shuffle(fs)
    label = raga2idx[raga_key]
    if len(fs) >= 3:
        test_files.append((fs[0], label))
        val_files.append((fs[1], label))
        train_files.extend((f, label) for f in fs[2:])
    else:
        train_files.extend((f, label) for f in fs)

all_files = train_files + val_files + test_files
print(f"train={len(train_files)}  val={len(val_files)}  test={len(test_files)}")

In [ ]:
# --- Precompute (only runs for files not already in the attached cache) —
# identical logic to raga_classifier_pitch_contour.ipynb, kept in sync so
# this notebook is self-sufficient even with no cache attached.
#
# Stride now depends on the file's split (2026-09-18, ported from the GRU
# notebook — see config cell): train_files get TRAIN_SEGMENT_STRIDE
# (overlapping), val_files + test_files get EVAL_SEGMENT_STRIDE
# (non-overlapping, for independent majority-vote votes). _cache_path now
# takes the stride explicitly instead of reading one global value. ---
import numpy as np
import librosa
import hashlib
from concurrent.futures import ProcessPoolExecutor, as_completed

def _cache_path(path, stride):
    config_key = (AUDIO_SAMPLE_RATE, MELODY_HOP_SIZE, QUANT_K, VOCAB_SIZE, INPUT_LENGTH,
                  stride, MIN_VOICED_FRACTION, MAX_SEGMENTS_PER_FILE, MAX_ANALYZE_SECONDS,
                  INTRO_SKIP_SECONDS, OUTRO_SKIP_SECONDS)
    h = hashlib.md5(f"{path}|{config_key}".encode()).hexdigest()
    return os.path.join(SEGMENTS_CACHE_DIR, f"{h}.npy")

def _precompute_one(args):
    (path, sr, hop_size, quant_k, vocab_size, input_length, stride, min_voiced_frac,
     max_segs, max_analyze_secs, intro_skip_secs, outro_skip_secs, out_path) = args
    if os.path.exists(out_path):
        return path, "cached", None
    try:
        import essentia.standard as es

        y, _ = librosa.load(path, sr=sr, mono=True)
        total_len = len(y)
        intro_skip = int(intro_skip_secs * sr)
        outro_skip = int(outro_skip_secs * sr)
        usable_start, usable_end = intro_skip, total_len - outro_skip
        if usable_end - usable_start < sr * 5:
            usable_start, usable_end = 0, total_len

        max_len = int(max_analyze_secs * sr)
        if usable_end - usable_start > max_len:
            usable_end = usable_start + max_len

        y = y[usable_start:usable_end].astype(np.float32)

        tonic = es.TonicIndianArtMusic(sampleRate=sr)(y)
        pitch, _conf = es.PredominantPitchMelodia(sampleRate=sr, hopSize=hop_size)(y)

        voiced = pitch > 0
        tokens = np.zeros(len(pitch), dtype=np.int32)
        cents = 1200.0 * np.log2(pitch[voiced] / tonic)
        tokens[voiced] = np.clip(np.round(cents * (quant_k / 100.0)), 0, vocab_size - 1).astype(np.int32)

        segments = []
        for s in range(0, max(1, len(tokens) - input_length + 1), stride):
            chunk = tokens[s:s + input_length]
            if len(chunk) < input_length // 2:
                continue
            if voiced[s:s + len(chunk)].mean() < min_voiced_frac:
                continue
            if len(chunk) < input_length:
                chunk = np.pad(chunk, (0, input_length - len(chunk)))
            segments.append(chunk.astype(np.int16))
            if len(segments) >= max_segs:
                break

        if not segments:
            best_start, best_frac = 0, -1.0
            for s in range(0, max(1, len(tokens) - input_length + 1), stride):
                frac = voiced[s:s + input_length].mean()
                if frac > best_frac:
                    best_frac, best_start = frac, s
            chunk = tokens[best_start:best_start + input_length]
            if len(chunk) < input_length // 2:
                return path, "empty", None
            if len(chunk) < input_length:
                chunk = np.pad(chunk, (0, input_length - len(chunk)))
            segments.append(chunk.astype(np.int16))

        np.save(out_path, np.stack(segments))
        return path, "done", None
    except Exception as e:
        return path, "error", str(e)

def _make_tasks(files, stride):
    return [(p, AUDIO_SAMPLE_RATE, MELODY_HOP_SIZE, QUANT_K, VOCAB_SIZE, INPUT_LENGTH, stride,
              MIN_VOICED_FRACTION, MAX_SEGMENTS_PER_FILE, MAX_ANALYZE_SECONDS,
              INTRO_SKIP_SECONDS, OUTRO_SKIP_SECONDS, _cache_path(p, stride)) for p, _ in files]

tasks = _make_tasks(train_files, TRAIN_SEGMENT_STRIDE) + _make_tasks(val_files + test_files, EVAL_SEGMENT_STRIDE)
already_cached = sum(1 for t in tasks if os.path.exists(t[-1]))
print(f"{already_cached}/{len(tasks)} already cached (resumed) — {len(tasks) - already_cached} left to process")

t0 = time.time()
done, errors = 0, []
with ProcessPoolExecutor(max_workers=PRECOMPUTE_WORKERS) as ex:
    futures = [ex.submit(_precompute_one, t) for t in tasks]
    for fut in as_completed(futures):
        path, status, err = fut.result()
        if status == "error":
            errors.append((path, err))
            print(f"  ERROR on {path}: {err}")
        done += 1
        if done % 20 == 0 or done == len(tasks):
            elapsed = time.time() - t0
            rate = elapsed / max(1, done - already_cached) if done > already_cached else None
            remaining = len(tasks) - done
            eta = f"{rate * remaining / 60:.1f}min" if rate else "n/a (still warming up)"
            print(f"  {done}/{len(tasks)} processed, {elapsed/60:.1f}min elapsed, ETA for rest: {eta}")

print(f"Precompute finished: {done} processed, {len(errors)} errors.")
if errors:
    print("Files with errors (will be skipped below):", [e[0] for e in errors])

In [ ]:
# --- Train one CategoricalHMM per raga on its own training segments
# (Tansen-style). hmmlearn wants a single concatenated observation array
# plus a `lengths` list marking where each individual sequence starts.
#
# REWRITTEN 2026-09-18 after a live run showed a real, serious problem: 10
# of 61 models took 11,778s (~1,178s/model average) for the SMALLEST
# classes trained first — extrapolating to all 61 (the larger, capped
# classes still to come cost strictly more per model) implies 20+ hours of
# HMM training alone, on top of the ~3.65h precompute already spent. That's
# very likely to exceed any Kaggle session's time limit — and the old code
# only pickled `models` ONCE, after the entire loop finished, so a session
# getting killed mid-run meant losing every model trained so far, including
# ones that took 15-20 minutes each.
#
# Two real fixes:
# 1. PARALLELIZE the independent per-class fits across processes
#    (ProcessPoolExecutor, same tool + worker count already proven by the
#    precompute step). See the config cell for why this is the right lever
#    instead of GPU/TPU: hmmlearn has no GPU path, and this workload is
#    embarrassingly parallel across classes with zero shared state, so
#    running known-correct fit() calls concurrently carries none of the
#    correctness risk a from-scratch GPU Baum-Welch rewrite would.
# 2. INCREMENTAL, ATOMIC checkpointing: re-save `models` to MODELS_PATH
#    after EVERY completed fit (write to a .tmp file then os.replace, so a
#    kill mid-write can't leave a corrupt file for resume logic to choke
#    on), plus EXISTING_MODELS_INPUT to skip already-trained classes on a
#    resumed run. A session timeout now costs at most whatever's still
#    in-flight (at most HMM_TRAIN_WORKERS models), not everything.
# A crashed/OOM-killed worker is also caught at the fut.result() call below
# (not just exceptions raised inside the worker) so one bad class can't take
# the whole cell down — it's simply left untrained for a future resume.
#
# MAX_SEGMENTS_PER_CLASS also lowered 300 -> 150 as a second, complementary
# safety margin on worst-case per-model cost for the larger classes still to
# come (the smallest classes already run were nowhere near either cap, so
# this doesn't change anything already measured). 150 segments x 5000
# tokens = 750K observations is still a lot of data for a 12-state,
# 209-symbol categorical HMM — but this hasn't been evidence-checked against
# accuracy, so treat it as a safety margin, not a proven-safe cut.
#
# _load_cached now takes stride explicitly (2026-09-18, train/eval windowing
# decoupled — see config cell): training always reads TRAIN_SEGMENT_STRIDE's
# (overlapping) cache; the held-out test cell reads EVAL_SEGMENT_STRIDE's
# (non-overlapping) cache instead. ---
from hmmlearn import hmm
from collections import defaultdict as _dd
import pickle

MAX_SEGMENTS_PER_CLASS = 150  # bounds worst-case Baum-Welch cost per model

def _load_cached(path, stride):
    cache_path = _cache_path(path, stride)
    if not os.path.exists(cache_path):
        return None
    return np.load(cache_path)

train_segments_by_class = _dd(list)  # label -> list of 1D int arrays
skipped = 0
for path, label in train_files:
    segs = _load_cached(path, TRAIN_SEGMENT_STRIDE)
    if segs is None:
        skipped += 1
        continue
    for seg in segs:
        train_segments_by_class[label].append(seg.astype(np.int64))
if skipped:
    print(f"(skipping {skipped} train files with no cache)")

total_train_segments = sum(len(v) for v in train_segments_by_class.values())
print(f"{total_train_segments} total training segments across {len(train_segments_by_class)} ragas")

# Class priors computed from TRUE segment counts (before any capping below),
# so the Bayes decision rule still reflects the dataset's real imbalance.
class_priors = {
    label: len(segs) / total_train_segments
    for label, segs in train_segments_by_class.items()
}

rng = random.Random(SEED)
capped = 0
for label, segs in train_segments_by_class.items():
    if len(segs) > MAX_SEGMENTS_PER_CLASS:
        train_segments_by_class[label] = rng.sample(segs, MAX_SEGMENTS_PER_CLASS)
        capped += 1
if capped:
    print(f"Capped {capped} ragas to {MAX_SEGMENTS_PER_CLASS} segments each (reproducibly subsampled) "
          f"to bound worst-case per-model training time.")

# --- Resume support: load already-trained models from a prior (interrupted)
# run, so this run only trains what's left. ---
models = {}
if EXISTING_MODELS_INPUT:
    if not os.path.isfile(EXISTING_MODELS_INPUT):
        raise RuntimeError(f"EXISTING_MODELS_INPUT={EXISTING_MODELS_INPUT!r} not found.")
    with open(EXISTING_MODELS_INPUT, "rb") as f:
        _prior = pickle.load(f)
    models = _prior["models"]
    print(f"Resumed {len(models)} already-trained models from a prior run: "
          f"{sorted(idx2raga[l]['raga'] for l in models)}")
else:
    print("No existing models attached — training all classes from scratch.")

remaining = [(label, segs) for label, segs in train_segments_by_class.items() if label not in models]
# Smallest classes first — fast, frequent progress lines right away rather
# than possibly sitting on the biggest, slowest class first with nothing to show.
remaining.sort(key=lambda kv: len(kv[1]))
print(f"{len(remaining)}/{len(train_segments_by_class)} classes left to train "
      f"({len(train_segments_by_class) - len(remaining)} already done).")

def _train_one_hmm(args):
    label, segs, n_states, n_iter, seed, vocab_size = args
    from hmmlearn import hmm as _hmm  # imported inside the worker, same
                                       # discipline as essentia in the
                                       # precompute cell
    t0 = time.time()
    X = np.concatenate(segs).reshape(-1, 1)
    lengths = [len(s) for s in segs]
    model = _hmm.CategoricalHMM(n_components=n_states, n_iter=n_iter,
                                 random_state=seed, n_features=vocab_size)
    try:
        model.fit(X, lengths)
        return label, model, time.time() - t0, None
    except Exception as e:
        return label, None, time.time() - t0, str(e)

def _save_models_atomic():
    tmp_path = MODELS_PATH + ".tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump({"models": models, "class_priors": class_priors, "idx2raga": idx2raga}, f)
    os.replace(tmp_path, MODELS_PATH)  # atomic on the same filesystem — a kill
                                        # mid-write can't leave a corrupt file
                                        # behind for a future resume to choke on

t_total = time.time()
n_attempted, n_trained_ok = 0, 0
if remaining:
    tasks = [(label, segs, N_HMM_STATES, HMM_ITER, SEED, VOCAB_SIZE) for label, segs in remaining]
    with ProcessPoolExecutor(max_workers=HMM_TRAIN_WORKERS) as ex:
        futures = {ex.submit(_train_one_hmm, t): t[0] for t in tasks}
        for fut in as_completed(futures):
            label = futures[fut]
            raga_name = idx2raga[label]["raga"]
            n_attempted += 1
            try:
                _, model, fit_time, err = fut.result()
            except Exception as e:
                # Worker crashed/was OOM-killed outright (not a caught
                # exception inside _train_one_hmm) — don't let it take the
                # whole cell down. This class stays untrained and will be
                # picked up by a future EXISTING_MODELS_INPUT-based resume.
                print(f"[{n_attempted}/{len(remaining)}] '{raga_name}' -> WORKER CRASHED: {e}", flush=True)
                continue
            if err:
                print(f"[{n_attempted}/{len(remaining)}] '{raga_name}' -> WARNING: "
                      f"failed to fit ({fit_time:.0f}s): {err}", flush=True)
                continue
            models[label] = model
            n_trained_ok += 1
            conv_info = f"iters={model.monitor_.iter}, converged={model.monitor_.converged}"
            print(f"[{n_attempted}/{len(remaining)}] '{raga_name}' -> done in {fit_time:.0f}s "
                  f"({conv_info}), total elapsed this run: {(time.time()-t_total)/60:.1f}min", flush=True)
            _save_models_atomic()  # re-checkpoint after EVERY completed model
else:
    print("Nothing left to train — all classes already resumed from EXISTING_MODELS_INPUT.")

print(f"Done: {len(models)}/{len(ragas)} raga HMMs trained successfully overall "
      f"({n_trained_ok} newly trained this run out of {n_attempted} attempted) "
      f"in {(time.time()-t_total)/60:.1f}min this run.")
_save_models_atomic()
print(f"Saved trained models -> {MODELS_PATH}")

In [ ]:
# --- Held-out TEST evaluation: per-segment AND per-file majority-vote
# accuracy, matching the other two notebooks' methodology exactly so all
# three are directly comparable. Classification = Bayes rule: argmax over
# (log-likelihood under model_k + log(prior_k)). Also saves the full
# per-segment log-likelihood matrix for later use in an ensemble with the
# GRU's predictions.
# Uses EVAL_SEGMENT_STRIDE (non-overlapping) segments (2026-09-18, ported
# from the GRU notebook) — majority voting needs independent votes; see the
# config cell for the real evidence (41.0% -> 32.8% regression there) that
# forced this. ---
import math

log_priors = {label: math.log(p) for label, p in class_priors.items()}
fitted_labels = sorted(models.keys())

per_class_correct = defaultdict(int)
per_class_total = defaultdict(int)
seg_correct, seg_total = 0, 0
file_correct, file_total = 0, 0
all_test_loglik = []   # list of (file_path, true_label, {label: loglik per segment})

t0 = time.time()
for path, label in test_files:
    segs = _load_cached(path, EVAL_SEGMENT_STRIDE)
    if segs is None:
        continue
    seg_preds = []
    seg_logliks = []
    for seg in segs:
        x = seg.astype(np.int64).reshape(-1, 1)
        scores = {k: models[k].score(x) + log_priors[k] for k in fitted_labels}
        seg_logliks.append(scores)
        pred = max(scores, key=scores.get)
        seg_preds.append(pred)
        seg_total += 1
        if pred == label:
            seg_correct += 1
    all_test_loglik.append((path, label, seg_logliks))

    majority = max(set(seg_preds), key=seg_preds.count)
    file_total += 1
    per_class_total[label] += 1
    if majority == label:
        file_correct += 1
        per_class_correct[label] += 1

print(f"Scored {file_total} test files in {time.time()-t0:.0f}s")
print(f"Per-segment test accuracy: {seg_correct/max(1,seg_total):.3f}  ({seg_correct}/{seg_total})")
print(f"Per-file majority-vote test accuracy: {file_correct/max(1,file_total):.3f}  ({file_correct}/{file_total})")
print()
print("Per-class (majority-vote) breakdown:")
for label in sorted(per_class_total):
    print(f"  {idx2raga[label]['raga']:35s} {per_class_correct[label]}/{per_class_total[label]}")

In [ ]:
import json

with open("/kaggle/working/hmm_test_logliks.pkl", "wb") as f:
    pickle.dump(all_test_loglik, f)

with open("/kaggle/working/hmm_results.json", "w") as f:
    json.dump({
        "test_segment_acc": seg_correct / max(1, seg_total),
        "test_majority_vote_acc": file_correct / max(1, file_total),
        "num_ragas": len(ragas),
        "num_fitted_models": len(models),
        "n_hmm_states": N_HMM_STATES,
    }, f, indent=2)

print("Saved: hmm_models.pkl (61 trained HMMs + priors), hmm_test_logliks.pkl "
      "(per-segment log-likelihoods, for a later ensemble with the GRU), hmm_results.json")
print("Click 'Save Version' now to persist these for comparison / reuse.")